[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/06_linear_regression/01_linear_regression_one_variable.ipynb)

# Bài 01 — Hồi quy tuyến tính đơn biến (Linear Regression with One Variable)

Bài viết đầu tiên về các thuật toán trong Machine Learning, chúng ta sẽ cùng tìm hiểu thuật toán **Linear Regression** (hồi quy tuyến tính). Linear Regression là một trong những thuật toán cơ bản nhất của Machine Learning — một thuật toán **supervised** (học có giám sát) trong đó quan hệ giữa đầu vào và đầu ra được mô tả bởi một **hàm tuyến tính**.

Nhắc lại bài toán **regression** (hồi quy): đầu ra dự đoán là giá trị **liên tục** — giá nhà, doanh số, nhiệt độ...

<img src="images/linear-regression.png" style="width:45%;height:45%;">

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/06_linear_regression

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from ml_course.visualization import plot_loss_contour, set_style

set_style()

## 1. Giới thiệu bài toán hồi quy tuyến tính đơn biến

Hồi quy tuyến tính đơn biến dùng khi muốn dự đoán đầu ra $y$ từ **một** đầu vào $x$. Hàm giả thuyết (hypothesis):

$$\hat{y} = h_{\theta}(x) = \theta_0 + \theta_1 x$$

- Đây là phương trình đường thẳng với 2 tham số cần tìm: $\theta_0$ (hệ số chặn — *intercept*, còn gọi là bias $b$) và $\theta_1$ (độ dốc — *slope*, trọng số $w$).
- Ta muốn $\hat{y} \approx y$, với $y$ là giá trị thực.

**Ví dụ:** giả sử có bộ dữ liệu như sau:

<img src="images/linear-ex.png" style="width:35%">

Chọn ngẫu nhiên $\theta_0 = 2$, $\theta_1 = 2$ thì $h_{\theta}(x) = 2 + 2x$. Với $x = 0$: $\hat{y}(0) = 2$, khác xa $y(0) = 4$. Ta muốn tìm $\theta_0, \theta_1$ sao cho sai lệch giữa $\hat{y}$ và $y$ là **nhỏ nhất trên toàn bộ dữ liệu**.

## 2. Hàm mất mát — Cost Function

Hàm mất mát (cost function / loss function) đánh giá mức độ sai lệch giữa dự đoán $\hat{y}$ và giá trị thực $y$. Với hồi quy, ta dùng **sai số bình phương trung bình**:

$$J(\theta_0, \theta_1) = \frac{1}{2m}\sum_{i=1}^{m}\left(h_\theta(x^{(i)}) - y^{(i)}\right)^2$$

*Chú thích:*
- $m$ là số quan sát (số mẫu dữ liệu) — tài liệu khác thường ký hiệu $n$.
- $\frac{1}{m}\sum_{i=1}^m$ là lấy trung bình.
- Hệ số $\frac{1}{2}$ giúp triệt tiêu số 2 khi lấy đạo hàm — không ảnh hưởng tới điểm tối ưu.

Mục tiêu: tìm $\theta_0, \theta_1$ làm $J$ **nhỏ nhất**. Về hình học, mỗi số hạng $(h_\theta(x^{(i)}) - y^{(i)})^2$ là bình phương **khoảng cách theo chiều dọc** (residual) từ điểm dữ liệu tới đường thẳng *(bổ sung)*:

In [ ]:
data = np.genfromtxt("../../data/sample/my_house_price_prediction.csv", delimiter=",")
X = data[:, 0]            # diện tích nhà (× 100 m²)
y = data[:, 1]            # giá nhà (chục lượng vàng) — lấy cột dạng 1 chiều (n,)
m = len(y)


def cost_function(X, y, theta0, theta1):
    return np.sum((theta0 + theta1 * X - y) ** 2) / (2 * len(y))


fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, (t0, t1) in zip(axes, [(2.0, 0.2), (1.0, 1.1)]):
    y_hat = t0 + t1 * X
    ax.scatter(X, y, zorder=3)
    xs = np.linspace(X.min(), X.max(), 50)
    ax.plot(xs, t0 + t1 * xs, "r")
    ax.vlines(X, np.minimum(y, y_hat), np.maximum(y, y_hat), colors="gray", linestyles=":")
    ax.set_title(f"θ0={t0}, θ1={t1}: J = {cost_function(X, y, t0, t1):.3f}")
    ax.set_xlabel("Diện tích (× 100 m²)")
    ax.set_ylabel("Giá (chục lượng vàng)")
plt.show()

Đường bên phải có các residual (đoạn chấm) ngắn hơn → $J$ nhỏ hơn → khớp dữ liệu tốt hơn.

## 3. Gradient Descent

Gradient Descent là thuật toán lặp tối ưu hoá dùng trong rất nhiều bài toán Machine Learning / Deep Learning (chương 02, bài 05).

<img src="images/function-max-global.png" style="width:45%;height:45%;">

*Nhắc lại:*
- **Global minimum** là giá trị nhỏ nhất của hàm số; **local minimum** là cực tiểu địa phương.
- Global minimum là một trường hợp đặc biệt của local minimum.

<img src="images/gd-2.png" style="width:45%;height:45%;">

Gradient Descent lấy **đạo hàm** của cost function theo $\theta$ rồi di chuyển $\theta$ **ngược dấu** với đạo hàm để tiến dần về điểm cực tiểu.

Ví dụ hàm $y = x^2 - 5x + 100$, điểm hiện tại $x_t = 10$, điểm cực tiểu $x = 2.5$. Đạo hàm $y' = 2x - 5$, tại $x_t = 10$ thì $y' = 15 > 0$ → phải giảm $x$:

$$x_{new} = x_t - \alpha\,f'(x_t)$$

<img src="images/gd-3.png" style="width:45%;height:45%;">

với $\alpha$ là số dương gọi là **learning rate** (tốc độ học); dấu trừ biểu thị đi ngược dấu đạo hàm.

> 📝 *Đính chính so với bản gốc:* tại $x_t = 10$, $y' = 2\cdot 10 - 5 = 15$ (bản gốc ghi nhầm là 5).

## 4. Learning Rate

Learning rate quyết định độ dài mỗi bước. Chọn không phù hợp dẫn tới:

<img src="images/learning-rate.png" style="width:85%;height:85%;">

- **Quá nhỏ:** hội tụ rất chậm; với hàm không lồi có thể kẹt ở cực tiểu địa phương xấu (vị trí *local 01* trong hình dưới).
- **Vừa đủ:** hội tụ tới cực tiểu tốt.
- **Quá lớn:** dao động quanh cực tiểu hoặc phân kỳ.

<img src="images/lr-2.png" style="width:35%;height:35%;">

Kỹ thuật chọn learning rate được trình bày ở bài 03 (Gradient Descent tips).

## 5. Gradient Descent cho bài toán Linear Regression

Lấy đạo hàm riêng của $J$ theo $\theta_0$ và $\theta_1$ (chain rule — chương 02, bài 04):

$$\frac{\partial J}{\partial \theta_0} = \frac{1}{m}\sum_{i=1}^m\left(h_\theta(x^{(i)}) - y^{(i)}\right), \qquad \frac{\partial J}{\partial \theta_1} = \frac{1}{m}\sum_{i=1}^m\left(h_\theta(x^{(i)}) - y^{(i)}\right)x^{(i)}$$

Lặp lại cho đến khi hội tụ (cập nhật **đồng thời** cả hai tham số):

$$\theta_0 := \theta_0 - \alpha\frac{1}{m}\sum_{i=1}^m\left(h_\theta(x^{(i)}) - y^{(i)}\right) \qquad \theta_1 := \theta_1 - \alpha\frac{1}{m}\sum_{i=1}^m\left(h_\theta(x^{(i)}) - y^{(i)}\right)x^{(i)}$$

Điều kiện dừng có thể là: hết số vòng lặp định sẵn; giá trị cost đủ nhỏ; cost không giảm sau `n` vòng lặp.

**Mô tả quá trình hồi quy tuyến tính:**

<img src="images/gd.gif" style="width:60%;height:60%;">

## 6. Tổng kết nội dung

1. Hàm giả thuyết: $\hat{y} = h_{\theta}(x) = \theta_0 + \theta_1 x$
2. Hàm mất mát: $J(\theta_0, \theta_1) = \frac{1}{2m}\sum_{i=1}^m\left(h_{\theta}(x^{(i)}) - y^{(i)}\right)^2$
3. Gradient descent: cập nhật đồng thời $\theta_0, \theta_1$ ngược hướng đạo hàm.

## 7. Áp dụng cho bài toán dự đoán giá nhà đơn giản

Bộ dữ liệu nhà có 20 mẫu, mỗi mẫu gồm diện tích nhà $x$ (cột trái) và giá nhà $y$ (cột phải). Diện tích $x$ là **đặc trưng (feature)**, giá nhà $y$ là **nhãn (label / ground truth)**.

Ở đây ta dùng **toàn bộ dữ liệu** cho mỗi vòng lặp — gọi là **Batch Gradient Descent**. Các biến thể Mini-batch và Stochastic GD ở bài 04.

<img src="images/house-predict.png" style="width:55%;height:55%;">

In [ ]:
plt.scatter(X, y)
plt.xlabel("Diện tích nhà (× 100 m²)")
plt.ylabel("Giá nhà (chục lượng vàng)")
plt.show()

Hàm dự đoán, cập nhật tham số và train — viết rõ ràng bằng vòng lặp theo đúng công thức (bài 02 sẽ **vector hoá**):

In [ ]:
def predict(x, theta0, theta1):
    return theta0 + theta1 * x


def update_weights(X, y, theta0, theta1, learning_rate):
    theta0_deriv, theta1_deriv = 0.0, 0.0
    for i in range(len(y)):                          # tính tổng trước
        error = (theta0 + theta1 * X[i]) - y[i]
        theta0_deriv += error
        theta1_deriv += error * X[i]
    theta0 -= learning_rate * theta0_deriv / len(y)  # di chuyển ngược dấu đạo hàm
    theta1 -= learning_rate * theta1_deriv / len(y)
    return theta0, theta1


def train(X, y, theta0, theta1, learning_rate, iters):
    cost_history, path = [], [(theta0, theta1)]
    for _ in range(iters):
        theta0, theta1 = update_weights(X, y, theta0, theta1, learning_rate)
        cost_history.append(cost_function(X, y, theta0, theta1))
        path.append((theta0, theta1))
    return theta0, theta1, cost_history, np.array(path)


rng = np.random.default_rng(2020)                    # khởi tạo ngẫu nhiên, cố định seed để tái lập
theta0, theta1 = rng.normal(), rng.normal()
theta0, theta1, cost_history, path = train(X, y, theta0, theta1, learning_rate=0.01, iters=500)
print(f"θ0 = {theta0:.4f}, θ1 = {theta1:.4f}, J cuối = {cost_history[-1]:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(cost_history)
axes[0].set_xlabel("vòng lặp")
axes[0].set_ylabel("J(θ)")
axes[0].set_title("Cost giảm dần qua các vòng lặp")
axes[1].scatter(X, y)
xs = np.linspace(X.min(), X.max(), 50)
axes[1].plot(xs, predict(xs, theta0, theta1), color="r")
axes[1].set_title("Đường hồi quy tìm được")
plt.show()

Dự đoán giá căn nhà có diện tích 4.5 (× 100 m²):

In [ ]:
print("Giá nhà có diện tích 4.5 là:", round(predict(4.5, theta0, theta1), 3), "chục lượng vàng")

## 8. Nhìn quá trình học trên mặt cost $J(\theta_0, \theta_1)$ *(bổ sung)*

Vẽ contour của $J$ theo hai tham số và đường đi của gradient descent. Chú ý đường đi **cong**: các đường đồng mức là elip dẹt (chương 02, bài 05) — chuẩn hoá feature sẽ giúp GD đi thẳng hơn (bài 02).

In [ ]:
t0_star, t1_star = path[-1]
fig, ax = plt.subplots(figsize=(7, 5))
plot_loss_contour(lambda a, b: cost_function(X, y, a, b), (t0_star - 3, t0_star + 3), (t1_star - 1.5, t1_star + 1.5),
                  path=path[::10], ax=ax, labels=("θ0", "θ1"))
ax.set_title("Đường đi của Gradient Descent trên mặt J(θ0, θ1)")
plt.show()

## 9. Kiểm tra với nghiệm đóng và scikit-learn *(bổ sung)*

Với một biến, bài toán có **nghiệm đóng** (đặt đạo hàm bằng 0):

$$\theta_1 = \frac{\sum_i (x^{(i)} - \bar x)(y^{(i)} - \bar y)}{\sum_i (x^{(i)} - \bar x)^2} = r_{xy}\frac{s_y}{s_x}, \qquad \theta_0 = \bar y - \theta_1\bar x$$

— độ dốc chính là hệ số tương quan (chương 02, bài 08) nhân với tỉ lệ độ lệch chuẩn.

In [ ]:
from sklearn.linear_model import LinearRegression

t1_closed = np.sum((X - X.mean()) * (y - y.mean())) / np.sum((X - X.mean()) ** 2)
t0_closed = y.mean() - t1_closed * X.mean()
sk = LinearRegression().fit(X.reshape(-1, 1), y)
print(f"Gradient descent (500 vòng): θ0 = {theta0:.4f}, θ1 = {theta1:.4f}")
print(f"Nghiệm đóng                : θ0 = {t0_closed:.4f}, θ1 = {t1_closed:.4f}")
print(f"scikit-learn               : θ0 = {sk.intercept_:.4f}, θ1 = {sk.coef_[0]:.4f}")

Gradient descent sau 500 vòng **chưa hội tụ hẳn** về nghiệm tối ưu (θ0 còn lệch) — đúng như hình contour cho thấy: GD bị chậm theo hướng "dẹt". Bài 02 và 03 giải thích cách khắc phục (chuẩn hoá feature, chọn learning rate, normal equation).

## 🧠 Kiểm tra nhanh

1. Vì sao cost function có hệ số $\frac{1}{2}$?
2. Tại sao phải cập nhật $\theta_0$ và $\theta_1$ **đồng thời** (dùng giá trị cũ cho cả hai)?
3. Với learning rate quá lớn, đồ thị `cost_history` sẽ trông như thế nào?
4. Diễn giải $\theta_1$ trong bài toán giá nhà: tăng 100 m² thì giá thay đổi bao nhiêu?

## Tổng kết

Bài này cho cái nhìn đầu tiên về hồi quy tuyến tính đơn biến: hàm giả thuyết, hàm mất mát, gradient descent. Còn nhiều vấn đề cần đề cập: chọn learning rate, khởi tạo tham số, làm việc với nhiều feature, dữ liệu lớn, chuẩn hoá feature... — các bài tiếp theo sẽ lần lượt giải quyết.

## Tài liệu tham khảo

[1] [CS229 — Machine Learning Course](http://cs229.stanford.edu) · [2] [Learning rate of your neural network](https://www.jeremyjordan.me/nn-learning-rate/)

---
<sub>Bài gốc: *Linear Regression with One Variable* — [AIVN-Machine-Learning](https://github.com/thanhhff/AIVN-Machine-Learning), Tuần 2 (MIT License). Đã sửa: `y` lấy dạng 1 chiều, đạo hàm tại x=10; bổ sung: hình residual, contour J với đường đi GD, nghiệm đóng và so sánh scikit-learn.</sub>